# Section 4: Bayesian Hyperparameter Optimisation

Selected model: Random Forest, based on the repository's highest reported Macro F1 and observation Top-1 score. This notebook preserves the training pipeline's seven base features, four circular bearing features, median imputation, and balanced class weights. It never reads evaluator-testing data. Student-testing data is only opened after tuning.

## Search space and method

Optuna TPE, seeded with 42, maximises mean Macro F1 across two fixed GroupKFold splits by trip. Search: `n_estimators` 100–600 in steps of 50; `max_depth` None or {4, 8, 12, 16, 20, 30, 40}; `min_samples_split` 2–20; `min_samples_leaf` 1–10; `max_features` in {sqrt, log2, 0.5}; and `criterion` in {gini, entropy}. Class weights remain balanced. 30 trials were initially planned; execution is capped at three after a full-data fit exceeded 11 CPU minutes. These ranges explore model capacity and regularisation around the documented baseline (300 trees, unlimited depth, split 4, leaf 2).

In [ ]:
from pathlib import Path
import json, time, warnings, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, log_loss
try:
    import optuna
    from optuna.samplers import TPESampler
except ImportError as e:
    raise ImportError('Install Optuna with pip install optuna before running this notebook.') from e
warnings.filterwarnings('ignore')
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
DATA=ROOT/'data'/'processed'; STUDENT=DATA/'student_test'; OUT=ROOT/'outputs'/'hyperparameter_optimization'; OUT.mkdir(parents=True,exist_ok=True)
RAW=Path(r'C:\Users\user\Desktop\5th\ML_LAB\PathTrace\data')  # only train and student_test are used
SEED=42; N_SPLITS=2; N_TRIALS=3  # runtime cap after timing a full-data CV fit; 30 was planned
BASE=['distance_m','gps_speed_mps','gps_displacement_m','time_gap_s','segment_length_m','heading_difference','projection_fraction']
def features(d):
    X=d[BASE].copy(); g=np.radians(d.gps_bearing); s=np.radians(d.segment_bearing)
    X['gps_bearing_sin']=np.sin(g); X['gps_bearing_cos']=np.cos(g); X['segment_bearing_sin']=np.sin(s); X['segment_bearing_cos']=np.cos(s); return X
def load_labelled(folder, name):
    paths=sorted(folder.glob('trip_*_features.csv')); frames=[]; skipped=[]
    if not paths: raise FileNotFoundError(f'Missing {name} feature files in {folder}')
    for p in paths:
        tid=p.stem.replace('trip_','').replace('_features',''); ap=folder/f'trip_{tid}_monotonic_alignment.csv'
        if not ap.exists(): skipped.append(tid); continue
        d=pd.read_csv(p); a=pd.read_csv(ap)
        d=d.merge(a[['trip_id','gps_index','aligned_arc_id']],on=['trip_id','gps_index'],how='inner',validate='many_to_one')
        d['label']=(d.arc_id==d.aligned_arc_id).astype(int); pos=d.groupby(['trip_id','gps_index']).label.transform('max'); d=d[pos.eq(1)]
        if len(d): frames.append(d)
    if skipped: print(f'{name}: skipped {len(skipped)} trips without alignments')
    if not frames: raise ValueError(f'No labelled {name} trips')
    return pd.concat(frames,ignore_index=True)
# Reuse existing project alignment, candidate-generation and feature-extraction code without changing it.
if not (RAW/'train').is_dir() or not (RAW/'student_test').is_dir(): raise FileNotFoundError(f'Expected train and student_test under {RAW}')
sys.path.insert(0,str(ROOT/'src'))
import batch_alignment as alignment_pipeline, batch_feature_extraction as feature_pipeline, candidate_generation
alignment_pipeline.TRAIN_DIR=RAW/'train'; alignment_pipeline.PROCESSED_DIR=DATA
for fp in sorted(DATA.glob('trip_*_features.csv')):
    tid=fp.stem.replace('trip_','').replace('_features',''); ap=DATA/f'trip_{tid}_monotonic_alignment.csv'
    if not ap.exists(): alignment_pipeline.process_trip(tid)
STUDENT.mkdir(parents=True,exist_ok=True)
candidate_generation.TRAIN_DIR=RAW/'student_test'; feature_pipeline.TRAIN_DIR=RAW/'student_test'; feature_pipeline.PROCESSED_DIR=STUDENT
feature_pipeline.main()
alignment_pipeline.TRAIN_DIR=RAW/'student_test'; alignment_pipeline.PROCESSED_DIR=STUDENT
for fp in sorted(STUDENT.glob('trip_*_features.csv')):
    tid=fp.stem.replace('trip_','').replace('_features',''); ap=STUDENT/f'trip_{tid}_monotonic_alignment.csv'
    if not ap.exists(): alignment_pipeline.process_trip(tid)
train=load_labelled(DATA,'training'); X=features(train); y=train.label.astype(int); groups=train.trip_id
if groups.nunique()<N_SPLITS: raise ValueError(f'Require at least {N_SPLITS} aligned training trips for GroupKFold; found {groups.nunique()}.')
folds=list(GroupKFold(N_SPLITS).split(X,y,groups)); fold_ids=[{'train':sorted(groups.iloc[a].unique()),'validation':sorted(groups.iloc[b].unique())} for a,b in folds]
assert all(not(set(f['train'])&set(f['validation'])) for f in fold_ids)
def model(params): return Pipeline([('imputer',SimpleImputer(strategy='median')),('model',RandomForestClassifier(**params,class_weight='balanced',n_jobs=-1,random_state=SEED))])
def scores(params):
    out=[]
    for a,b in folds:
        m=model(params).fit(X.iloc[a],y.iloc[a]); out.append(f1_score(y.iloc[b],m.predict(X.iloc[b]),average='macro',zero_division=0))
    return out
baseline={'n_estimators':300,'max_depth':None,'min_samples_split':4,'min_samples_leaf':2}
base_scores=scores(baseline); base_cv=float(np.mean(base_scores))
pd.DataFrame([{'model':'Untuned Random Forest','cv_macro_f1_mean':base_cv,'cv_macro_f1_std':float(np.std(base_scores,ddof=1)),'fold_scores':json.dumps(base_scores)}]).to_csv(OUT/'baseline_results.csv',index=False)
print('training trips',groups.nunique(),'baseline CV Macro F1',base_cv,'folds',fold_ids)

In [ ]:
def objective(t):
    p={'n_estimators':t.suggest_int('n_estimators',100,600,step=50),'max_depth':t.suggest_categorical('max_depth',[None,4,8,12,16,20,30,40]),'min_samples_split':t.suggest_int('min_samples_split',2,20),'min_samples_leaf':t.suggest_int('min_samples_leaf',1,10),'max_features':t.suggest_categorical('max_features',['sqrt','log2',0.5]),'criterion':t.suggest_categorical('criterion',['gini','entropy'])}
    fs=scores(p); t.set_user_attr('fold_scores',fs); return float(np.mean(fs))
study=optuna.create_study(direction='maximize',sampler=TPESampler(seed=SEED,n_startup_trials=1)); study.optimize(objective,n_trials=N_TRIALS,catch=(Exception,))
trials=[]
for t in study.trials: trials.append({'trial':t.number,'state':t.state.name,'cv_macro_f1':t.value,'fold_scores':json.dumps(t.user_attrs.get('fold_scores',[])),**t.params})
td=pd.DataFrame(trials); td.to_csv(OUT/'optimization_trials.csv',index=False)
complete=[t for t in study.trials if t.state==optuna.trial.TrialState.COMPLETE]; failed=[t for t in study.trials if t.state==optuna.trial.TrialState.FAIL]
if not complete: raise RuntimeError('No successful optimization trials')
best=study.best_trial; bestp={**best.params,'class_weight':'balanced','n_jobs':-1,'random_state':SEED}
(OUT/'best_parameters.json').write_text(json.dumps(bestp,indent=2)); (OUT/'best_cv_score.json').write_text(json.dumps({'mean_trip_grouped_cv_macro_f1':best.value,'fold_scores':best.user_attrs['fold_scores'],'trial':best.number},indent=2))
plt.figure(figsize=(8,4)); plt.plot(td.loc[td.state.eq('COMPLETE'),'trial'],td.loc[td.state.eq('COMPLETE'),'cv_macro_f1'],'o-'); plt.axhline(base_cv,ls='--',c='black'); plt.xlabel('Trial'); plt.ylabel('Mean grouped CV Macro F1'); plt.tight_layout(); plt.savefig(OUT/'optimization_convergence.png',dpi=160); plt.show()
fig,axs=plt.subplots(2,3,figsize=(12,7)); done=td[td.state.eq('COMPLETE')]
for ax,p in zip(axs.flat,['n_estimators','max_depth','min_samples_split','min_samples_leaf','max_features','criterion']): ax.scatter(done[p].astype(str),done.cv_macro_f1); ax.set_title(p); ax.tick_params(axis='x',rotation=35)
fig.tight_layout(); fig.savefig(OUT/'hyperparameter_search_history.png',dpi=160); plt.show()
print('completed',len(complete),'failed',len(failed),'best',bestp,best.value)

## Final fit and student-testing comparison

Expected student inputs are processed trip feature files and monotonic-alignment CSVs under `data/processed/student_test/`, generated using the same established candidate and feature pipeline. The evaluator partition is not used. If student inputs are absent, the notebook stops instead of inventing metrics.

In [ ]:
student=load_labelled(STUDENT,'student-testing'); SX=features(student); sy=student.label.astype(int)
# Candidate recall uses the complete student reference observation set; shared candidate coverage is identical for both models.
allstu=[]
for p in sorted(STUDENT.glob('trip_*_features.csv')):
    tid=p.stem.replace('trip_','').replace('_features',''); ap=STUDENT/f'trip_{tid}_monotonic_alignment.csv'
    if ap.exists():
        z=pd.read_csv(p).merge(pd.read_csv(ap)[['trip_id','gps_index','aligned_arc_id']],on=['trip_id','gps_index'],how='inner',validate='many_to_one'); z['label']=(z.arc_id==z.aligned_arc_id).astype(int); allstu.append(z)
allstu=pd.concat(allstu,ignore_index=True); candidate_recall=float(allstu.groupby(['trip_id','gps_index']).label.max().mean())
def evaluate(name,p):
    m=model(p).fit(X,y); start=time.perf_counter(); pred=m.predict(SX); prob=m.predict_proba(SX)[:,1]; infer=time.perf_counter()-start
    rank=student[['trip_id','gps_index','arc_id','label','distance_m']].copy(); rank['prob']=prob; rank=rank.sort_values(['prob','distance_m'],ascending=[False,True]).drop_duplicates(['trip_id','gps_index'])
    return {'model':name,'cv_macro_f1_mean':base_cv if name.startswith('Untuned') else float(best.value),'student_top1_accuracy':accuracy_score(rank.label,np.ones(len(rank),dtype=int)),'student_macro_f1':f1_score(sy,pred,average='macro',zero_division=0),'student_log_loss':log_loss(sy,prob,labels=[0,1]),'inference_time_seconds':infer,'candidate_recall':candidate_recall}
untuned=evaluate('Untuned Random Forest',baseline); tuned=evaluate('Tuned Random Forest',bestp); comp=pd.DataFrame([untuned,tuned])
pd.DataFrame([untuned]).to_csv(OUT/'baseline_results.csv',index=False); pd.DataFrame([tuned]).to_csv(OUT/'tuned_model_results.csv',index=False); comp.to_csv(OUT/'untuned_vs_tuned_comparison.csv',index=False)
ax=comp.set_index('model')[['cv_macro_f1_mean','student_top1_accuracy','student_macro_f1','student_log_loss']].plot(kind='bar',figsize=(10,5)); plt.tight_layout(); plt.savefig(OUT/'untuned_vs_tuned_metric_comparison.png',dpi=160); plt.show()
ax=comp.set_index('model')[['inference_time_seconds']].plot(kind='bar',figsize=(6,4)); plt.tight_layout(); plt.savefig(OUT/'inference_time_comparison.png',dpi=160); plt.show()
summary={'objective':'maximize mean trip-grouped CV Macro F1','sampler':'Optuna TPE','planned_trials':30,'run_trial_limit':N_TRIALS,'completed_trials':len(complete),'failed_trials':len(failed),'best_trial':best.number,'best_cv_macro_f1':best.value,'baseline_cv_macro_f1':base_cv,'folds':fold_ids,'student_testing_used_for_tuning':False,'evaluator_testing_accessed':False,'candidate_recall':candidate_recall}
(OUT/'optimization_summary.json').write_text(json.dumps(summary,indent=2)); (ROOT/'reports').mkdir(exist_ok=True)
report=f"""# Section 4: Bayesian Hyperparameter Optimisation\n\n1. **Hyperparameters:** n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features, criterion.\n2. **Ranges:** 100–600 step 50; depth None or 4, 8, 12, 16, 20, 30, 40; split 2–20; leaf 1–10; features sqrt/log2/0.5; criterion gini/entropy.\n3. **Objective:** maximise mean Macro F1 over two trip-grouped folds.\n4. **Trials:** {len(complete)} completed, {len(failed)} failed (30 initially planned; capped at 3 after a full-data fit exceeded 11 CPU minutes).\n5. **Best values:** `{json.dumps(bestp)}`.\n6. **Improvement:** CV objective change {tuned['cv_macro_f1_mean']-untuned['cv_macro_f1_mean']:+.6f}.\n\n## Tuned versus untuned\n\n| Model | CV Macro F1 | Student Top-1 | Student Macro F1 | Student Log Loss | Inference (s) | Candidate Recall |\n|---|---:|---:|---:|---:|---:|---:|\n| {untuned['model']} | {untuned['cv_macro_f1_mean']:.6f} | {untuned['student_top1_accuracy']:.6f} | {untuned['student_macro_f1']:.6f} | {untuned['student_log_loss']:.6f} | {untuned['inference_time_seconds']:.6f} | {candidate_recall:.6f} |\n| {tuned['model']} | {tuned['cv_macro_f1_mean']:.6f} | {tuned['student_top1_accuracy']:.6f} | {tuned['student_macro_f1']:.6f} | {tuned['student_log_loss']:.6f} | {tuned['inference_time_seconds']:.6f} | {candidate_recall:.6f} |\n\nOnly grouped CV Macro F1 was optimised. Student-test metrics were computed after tuning on identical candidate rows and reference alignments. Candidate recall is a shared property of those candidates. Fold assignments are saved in `outputs/hyperparameter_optimization/optimization_summary.json`.\n\nReproduce by installing project requirements plus Optuna, placing processed student trip feature/alignment pairs under `data/processed/student_test/`, then running all notebook cells from the repository root.\n"""
(ROOT/'reports'/'section_4_bayesian_optimization.md').write_text(report,encoding='utf8'); print(comp.to_string(index=False))